In [1]:
%pip install opencv-python
%pip install open3d

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from copy import deepcopy
from sklearn.neighbors import NearestNeighbors
import cv2

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [3]:
def plot_point_clouds(pcd_list: list[o3d.geometry.PointCloud], 
                      azim: float=-60, 
                      elev:float=30) -> None:
    """
    Plot point clouds.

    Args:
        pcd_list: list of point cloud data objects
        azim: azimuth angle (deg)
        elev: elevation angle (deg)
    """
    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(111, projection='3d')
    color_fns = [plt.cm.viridis, plt.cm.inferno]
    for i, pcd in enumerate(pcd_list):
      points = np.asarray(pcd.points)
      colors = color_fns[i](points[:, 2] / points[:, 2].max())
      ax.scatter(points[:, 0], points[:, 1], points[:, 2], s=10, c=colors)
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.view_init(azim=azim, elev=elev)
    plt.show()

In [4]:
# Load full bunny point cloud
full_bunny = o3d.io.read_point_cloud("bun_zipper.ply")

# Load partial bunny point cloud
# Note: `pcd` refers to "point cloud data"
pcd = o3d.io.read_point_cloud("bun045.ply")

# pcd is an open3d data structure with different attributes
# You can access the 3D points representing the lidar data like this:
pcd_np = np.asarray(pcd.points)
print(pcd_np.shape)

(40097, 3)


In [5]:
_ = widgets.interact(
    plot_point_clouds,
    pcd_list=widgets.fixed([full_bunny]),
    azim=widgets.IntSlider(-90, min=-180, max=180, step=5, description="Azimuth"),
    elev=widgets.IntSlider(90, min=0, max=90, step=5, description="Elevation")
)

_ = widgets.interact(
    plot_point_clouds,
    pcd_list=widgets.fixed([pcd]),
    azim=widgets.IntSlider(-90, min=-180, max=180, step=5, description="Azimuth"),
    elev=widgets.IntSlider(90, min=0, max=90, step=5, description="Elevation")
)

interactive(children=(IntSlider(value=-90, description='Azimuth', max=180, min=-180, step=5), IntSlider(value=…

interactive(children=(IntSlider(value=-90, description='Azimuth', max=180, min=-180, step=5), IntSlider(value=…

In [6]:
_ = widgets.interact(
    plot_point_clouds,
    pcd_list=widgets.fixed([full_bunny, pcd]),
    azim=widgets.IntSlider(-90, min=-180, max=180, step=5, description="Azimuth"),
    elev=widgets.IntSlider(90, min=0, max=90, step=5, description="Elevation")
)

interactive(children=(IntSlider(value=-90, description='Azimuth', max=180, min=-180, step=5), IntSlider(value=…

In [7]:
def preprocess_point_cloud(pcd: o3d.geometry.PointCloud, 
                           voxel_size: float,
) -> tuple[o3d.geometry.PointCloud, o3d.pipelines.registration.Feature]:
    """
    Extract features from point cloud data.

    Args:
        pcd: point cloud data
        voxel_size: voxel size

    Returns:
        Down sampled point cloud and the "FPFH" features.
    """
    pcd_down_sampled = pcd.voxel_down_sample(voxel_size)

    radius_normal = voxel_size * 2
    pcd_down_sampled.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(radius=radius_normal, max_nn=30))

    radius_feature = voxel_size * 5
    pcd_fpfh = o3d.pipelines.registration.compute_fpfh_feature(
        pcd_down_sampled,
        o3d.geometry.KDTreeSearchParamHybrid(radius=radius_feature, max_nn=100))
    return pcd_down_sampled, pcd_fpfh

source, target = pcd, full_bunny
voxel_size = 0.01
source_down, source_fpfh = preprocess_point_cloud(source, voxel_size)
target_down, target_fpfh = preprocess_point_cloud(target, voxel_size)

In [8]:
def execute_global_registration(source_down: o3d.geometry.PointCloud, 
                                target_down: o3d.geometry.PointCloud, 
                                source_fpfh: o3d.pipelines.registration.Feature,
                                target_fpfh: o3d.pipelines.registration.Feature, 
                                voxel_size: float) -> np.ndarray:
    """
    Run RANSAC algorithm to perform global registration on the source and target
    point clouds.

    Args:
        source_down: down sampled source point cloud
        target_down: down sampled target point cloud
        source_fpfh: source point cloud FPFH features
        target_fpfh: target point cloud FPFH features
        voxel_size: voxel size

    Returns:
        Transformation matrix, T.
    """
    distance_threshold = voxel_size * 1.5
    print(":: RANSAC registration on downsampled point clouds.")
    print("   Since the downsampling voxel size is %.3f," % voxel_size)
    print("   we use a liberal distance threshold %.3f." % distance_threshold)
    result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
        source_down, target_down, source_fpfh, target_fpfh, False, distance_threshold)
    return result.transformation

T = execute_global_registration(source_down, target_down, source_fpfh, target_fpfh, voxel_size=0.01)
print("T shape:", T.shape)
print(T)

:: RANSAC registration on downsampled point clouds.
   Since the downsampling voxel size is 0.010,
   we use a liberal distance threshold 0.015.
T shape: (4, 4)
[[ 8.37072779e-01 -6.38693632e-02  5.43350594e-01 -4.33847601e-02]
 [ 3.97996948e-02  9.97639706e-01  5.59553481e-02  2.00763164e-04]
 [-5.45641959e-01 -2.52135109e-02  8.37639022e-01 -1.49771997e-02]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]


In [9]:
lidar_transformed = deepcopy(pcd).transform(T)

_ = widgets.interact(
    plot_point_clouds,
    pcd_list=widgets.fixed([full_bunny, lidar_transformed]),
    azim=widgets.IntSlider(-90, min=-180, max=180, step=5, description="Azimuth"),
    elev=widgets.IntSlider(90, min=0, max=90, step=5, description="Elevation")
)

interactive(children=(IntSlider(value=-90, description='Azimuth', max=180, min=-180, step=5), IntSlider(value=…

In [10]:
def best_fit_transform(A: np.ndarray, 
                       B: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Calculates the least-squares best-fit transform that maps corresponding 
    points A to B in m spatial dimensions.
    
    Args:
      A: (Na, m) array of corresponding points
      B: (Nb, m) array of corresponding points
      
    Returns:
      T: (m + 1, m + 1) homogeneous transformation matrix that maps A on to B
      R: (m, m) rotation matrix
      t: (m, 1) translation vector
    """
    # Get number of dimensions
    m = A.shape[1]

    # Translate points to their centroids
    centroid_A = np.mean(A, axis=0)
    centroid_B = np.mean(B, axis=0)
    AA = A - centroid_A
    BB = B - centroid_B

    # Rotation matrix
    H = np.dot(AA.T, BB)
    U, S, Vt = np.linalg.svd(H)
    R = np.dot(Vt.T, U.T)

    # Special reflection case
    if np.linalg.det(R) < 0:
       Vt[m - 1, :] *= -1
       R = np.dot(Vt.T, U.T)

    # Translation
    t = centroid_B.T - np.dot(R, centroid_A.T)

    # Homogeneous transformation
    T = np.identity(m + 1)
    T[:m, :m] = R
    T[:m, m] = t

    return T, R, t

In [21]:
def nearest_neighbor(src: np.ndarray, 
                     dst: np.ndarray, 
                     radius: float=0.01,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Find the nearest (Euclidean) neighbor in dst for each point in src
    
    Args:
        src: (Na, m) array of points
        dst: (Nb, m) array of points
        
    Returns:
        distances: Euclidean distances of the nearest neighbor
        indices: dst indices of the nearest neighbor
    """
    ##### YOUR CODE STARTS HERE #####
    # Use the NearestNeighbors class sklearn.neighbors to compute the nearest
    # neighbor of each point in the source dataset to the target dataset
    # Note that by using this class and its methods correctly, you should be
    # able to get the distance between a point and its nearest neighbor,
    # as well as the indices of the nearest neighbor in the target point cloud
    # 1. Khởi tạo đối tượng NearestNeighbors với n_neighbors=1 để tìm 1 điểm gần nhất
    neigh = NearestNeighbors(n_neighbors=1)
    # 2. Fit đối tượng với tập điểm target/destination (dst)
    neigh.fit(dst)
    # 3. Tìm khoảng cách và chỉ số (index) của điểm gần nhất trong dst cho từng điểm trong src
    distances, indices = neigh.kneighbors(src, return_distance=True)
    ###### YOUR CODE ENDS HERE ######
    return distances.ravel(), indices.ravel()

def icp(A: np.ndarray, 
        B: np.ndarray, 
        init_pose: np.ndarray=None, 
        max_iterations: int=20, 
        tolerance: float=0.001, 
        knn_radius: float=0.01) -> np.ndarray:
    """
    The Iterative Closest Point method: finds best-fit transform that maps points A 
    on to points B.
    
    Args:
        A: (Na, m) array of source points
        B: (Nb, m) array of destination points
        init_pose: (m+1, m+1) homogeneous transformation
        max_iterations: exit algorithm after max_iterations
        tolerance: convergence criteria
        knn_radius: k-nearest neighbors radius
        
    Returns:
        Homogeneous transformation that maps A onto B.
    """
    # Get number of points and point dimension
    Na, m = A.shape
    Nb = B.shape[0]

    # Make points homogeneous, copy them to maintain the originals
    src = np.ones((m + 1, Na))
    dst = np.ones((m + 1, Nb))
    src[:m, :] = np.copy(A.T)
    dst[:m, :] = np.copy(B.T)

    # Apply the initial pose estimation #ta thấy ở đây ban đầu RANSAC đưa point cloud A về gần B,
    #ICP từ đó mới hoạt động
    if init_pose is not None:
        src = np.dot(init_pose, src)

    prev_error = 0
    
    ##### YOUR CODE STARTS HERE #####
    # Write the loop for the ICP algorithm here
    # Hints:
    # - Use the functions `nearest_neighbor()` and `best_fit_transform()`
    # - src and dst matrices are defined above with shape (m+1, N), while the above
    #   two function expect matrices of shape (N,m)
    for i in range(max_iterations):
        distances, indices = nearest_neighbor(src[:m, :].T, dst[:m, :].T, radius=knn_radius)

        
        # Step 2: Tính ma trận biến đổi best-fit giữa src và các điểm tương ứng trong dst
        T, _, _ = best_fit_transform(src[:m, :].T, dst[:m, indices].T)

        # Step 3: Áp dụng biến đổi T lên tập điểm nguồn src, đổi src nên  cần chạy vòng for, chạy 
        # đến khi nào mà sai số nhỏ, cho src càng ngày càng gần dst
        src = np.dot(T, src)

        mean_error = np.mean(distances)
        if np.abs(prev_error - mean_error) < tolerance:
            break

        prev_error = mean_error
        
    ###### YOUR CODE ENDS HERE ######
    
    # Calculate final transformation
    T,_,_ = best_fit_transform(A, src[:m,:].T)
    print(distances)
    print(indices)
    return T

In [22]:

# Convert to numpy arrays
source_np = np.asarray(source_down.points)
target_np = np.asarray(target_down.points)

# Run ICP to get transformation matrix
T_icp = icp(source_np, target_np, T, max_iterations=20, tolerance=1e-5, knn_radius=1e-2)
print(T_icp)
# Transform point clouds
pcd_down_ICP = deepcopy(source_down).transform(T_icp) # transformed downsampled pcd
pcd_ICP = deepcopy(source).transform(T_icp) # transformed complete size pcd

# # Display result
# _ = widgets.interact(
#     plot_point_clouds,
#     pcd_list=widgets.fixed([full_bunny, pcd_ICP]),
#     azim=widgets.IntSlider(-90, min=-180, max=180, step=5, description="Azimuth"),
#     elev=widgets.IntSlider(90, min=0, max=90, step=5, description="Elevation")
# )

[0.00399075 0.00298772 0.00467171 0.00204506 0.00367771 0.00342539
 0.00272463 0.00253459 0.00224024 0.00232566 0.00264365 0.0025982
 0.00304562 0.00151309 0.00290647 0.00266339 0.00606597 0.00581274
 0.00427292 0.00402908 0.00155617 0.00480869 0.00437396 0.0040522
 0.0047582  0.00052941 0.00251578 0.00150794 0.00362046 0.00412961
 0.00228774 0.00100657 0.00358167 0.00499731 0.00223505 0.00238593
 0.00375531 0.00470727 0.00122003 0.00423561 0.00242194 0.00179076
 0.00302536 0.00334606 0.00226849 0.00151651 0.00541866 0.0017448
 0.0032664  0.00490754 0.003756   0.00054674 0.00266471 0.00353619
 0.00095087 0.00215411 0.00177045 0.00371157 0.00388556 0.00461155
 0.00427316 0.00353237 0.00374711 0.00123634 0.00214151 0.00350115
 0.00438573 0.00186567 0.00627482 0.00203856 0.00421536 0.00106578
 0.00265222 0.00215346 0.00271329 0.00179452 0.00147752 0.00286408
 0.00484828 0.00131767 0.00458943 0.00425109 0.00167963 0.00148368
 0.00053218 0.00180051 0.00279072 0.00381873 0.00374387 0.0015471

In [13]:
threshold = 0.01
max_iterations = 3000

##### YOUR CODE STARTS HERE #####
reg_p2p = o3d.pipelines.registration.registration_icp(pcd, full_bunny, threshold, T,
                                  o3d.pipelines.registration.TransformationEstimationPointToPoint(),
                                  criteria=o3d.pipelines.registration.ICPConvergenceCriteria(
                                      max_iteration=max_iterations))
###### YOUR CODE ENDS HERE ######

print(reg_p2p)
print("Transformation is:")
print(reg_p2p.transformation)

pcd_robust_ICP = deepcopy(pcd).transform(reg_p2p.transformation)

_ = widgets.interact(
    plot_point_clouds,
    pcd_list=widgets.fixed([full_bunny, pcd_robust_ICP]),
    azim=widgets.IntSlider(
        -90, min=-180, max=180, step=5, description="Azimuth"
    ),
    elev=widgets.IntSlider(90, min=0, max=90, step=5, description="Elevation"),
)

RegistrationResult with fitness=1.000000e+00, inlier_rmse=5.498974e-04, and correspondence_set size of 40097
Access transformation to get result.
Transformation is:
[[ 8.24852278e-01 -1.14125408e-02  5.65233115e-01 -5.18696133e-02]
 [ 3.59795944e-03  9.99881950e-01  1.49379226e-02 -2.41332425e-04]
 [-5.65336869e-01 -1.02878937e-02  8.24795965e-01 -1.07564780e-02]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]


interactive(children=(IntSlider(value=-90, description='Azimuth', max=180, min=-180, step=5), IntSlider(value=…

In [11]:
print("Load a ply point cloud, print it, and render it")
ply_point_cloud = o3d.data.PLYPointCloud()
pcd = o3d.io.read_point_cloud(ply_point_cloud.path)
print(pcd)
print(np.asarray(pcd.points))
o3d.visualization.draw_geometries([pcd],
                                  zoom=0.3412,
                                  front=[0.4257, -0.2125, -0.8795],
                                  lookat=[2.6172, 2.0475, 1.532],
                                  up=[-0.0694, -0.9768, 0.2024])

Load a ply point cloud, print it, and render it
PointCloud with 196133 points.
[[0.65234375 0.84686458 2.37890625]
 [0.65234375 0.83984375 2.38430572]
 [0.66737998 0.83984375 2.37890625]
 ...
 [2.00839925 2.39453125 1.88671875]
 [2.00390625 2.39488506 1.88671875]
 [2.00390625 2.39453125 1.88793314]]


In [15]:
print("Downsample the point cloud with a voxel of 0.05")
downpcd = pcd.voxel_down_sample(voxel_size=0.05)
o3d.visualization.draw_geometries([downpcd],
                                  zoom=0.3412,
                                  front=[0.4257, -0.2125, -0.8795],
                                  lookat=[2.6172, 2.0475, 1.532],
                                  up=[-0.0694, -0.9768, 0.2024])

Downsample the point cloud with a voxel of 0.05
